# Gold Careunit Daily Activity Fact

Build the careunit-level Gold fact for daily recorded hospital operational activity.

Output:
- `fact_careunit_daily_activity`

Grain:
- one row per `activity_date` + `careunit`

The fact summarises daily patient presence, arrivals and departures, transfer movement, ICU activity, and high-activity periods.

Cleaning and interval validity already completed in Silver are reused rather than repeated here.

This fact represents recorded operational activity, not true hospital capacity or staffed-bed occupancy.

In [0]:
from pyspark.sql import functions as F

SILVER_SCHEMA = "cdr_silver.processed_mimic_iv"
GOLD_SCHEMA = "cdr_gold.analytics_mimic_iv"

# Silver sources
transfers = spark.table(f"{SILVER_SCHEMA}.transfers")
admissions = spark.table(f"{SILVER_SCHEMA}.admissions")
icustays = spark.table(f"{SILVER_SCHEMA}.icustays")

# Gold dimensions
dim_date = spark.table(f"{GOLD_SCHEMA}.dim_date")
dim_careunit = spark.table(f"{GOLD_SCHEMA}.dim_careunit")

print("Operational activity fact source tables loaded.")

In [0]:
# Build operational careunit intervals from cleaned Silver transfers

operational_intervals = (
    transfers
    .filter(
        (F.col("eventtype").isin("admit", "transfer")) &
        (F.col("careunit_valid_flag") == 1) &
        (F.col("transfer_interval_valid_flag") == 1)
    )
    .join(
        admissions.select(
            "hadm_id",
            "dischtime"
        ),
        on="hadm_id",
        how="left"
    )
    .withColumn(
        "interval_days",
        (
            F.col("effective_outtime").cast("long")
            - F.col("intime").cast("long")
        ) / 86400.0
    )
    .withColumn(
        "extreme_timestamp_anomaly",
        (
            (F.col("interval_days") > 100) &
            (F.col("effective_outtime") > F.expr("dischtime + INTERVAL 30 DAYS"))
        )
    )
    .filter(
        (F.col("interval_days") > 0) &
        (~F.col("extreme_timestamp_anomaly"))
    )
)

display(
    operational_intervals.agg(
        F.count("*").alias("intervals"),
        F.countDistinct("hadm_id").alias("admissions"),
        F.countDistinct("subject_id").alias("patients"),
        F.countDistinct("careunit").alias("careunits"),
        F.sum("outtime_inferred_flag").alias("inferred_outtimes")
    )
)

In [0]:
# Expand careunit intervals into daily patient presence

daily_presence = (
    operational_intervals
    .withColumn(
        "presence_end_date",
        F.to_date(
            F.expr("effective_outtime - INTERVAL 1 SECOND")
        )
    )
    .withColumn(
        "activity_date",
        F.explode(
            F.sequence(
                F.to_date("intime"),
                F.col("presence_end_date")
            )
        )
    )
    .select(
        "subject_id",
        "hadm_id",
        "careunit",
        "activity_date"
    )
    .dropDuplicates(
        ["hadm_id", "careunit", "activity_date"]
    )
)

daily_presence_summary = (
    daily_presence
    .groupBy(
        "activity_date",
        "careunit"
    )
    .agg(
        F.countDistinct("subject_id").alias("active_patient_count"),
        F.countDistinct("hadm_id").alias("active_admission_count")
    )
)

display(
    daily_presence_summary.agg(
        F.count("*").alias("careunit_days"),
        F.countDistinct("careunit").alias("careunits"),
        F.min("activity_date").alias("min_date"),
        F.max("activity_date").alias("max_date")
    )
)

In [0]:
# Align admission count naming with the Gold specification

daily_presence_summary = (
    daily_presence_summary
    .withColumnRenamed(
        "active_admission_count",
        "distinct_admission_count"
    )
)

## Daily Careunit Movement

Create daily arrival and departure measures from the valid careunit intervals.

Interval starts represent arrivals into a careunit, while interval ends represent departures. Arrivals are separated into hospital admission and transfer activity.

In [0]:
# Create daily careunit arrival measures

daily_arrivals = (
    operational_intervals
    .withColumn(
        "activity_date",
        F.to_date("intime")
    )
    .groupBy(
        "activity_date",
        "careunit"
    )
    .agg(
        F.count("*").alias("patient_arrival_count"),

        F.sum(
            (F.col("eventtype") == "admit").cast("int")
        ).alias("admission_arrival_count"),

        F.sum(
            (F.col("eventtype") == "transfer").cast("int")
        ).alias("transfer_arrival_count")
    )
)

# Create daily careunit departure measures

daily_departures = (
    operational_intervals
    .withColumn(
        "activity_date",
        F.to_date("effective_outtime")
    )
    .groupBy(
        "activity_date",
        "careunit"
    )
    .agg(
        F.count("*").alias("patient_departure_count")
    )
)

# Validate movement totals

display(
    daily_arrivals.agg(
        F.sum("patient_arrival_count").alias("total_arrivals"),
        F.sum("admission_arrival_count").alias("admission_arrivals"),
        F.sum("transfer_arrival_count").alias("transfer_arrivals"),
        F.count("*").alias("arrival_careunit_days")
    )
)

display(
    daily_departures.agg(
        F.sum("patient_departure_count").alias("total_departures"),
        F.count("*").alias("departure_careunit_days")
    )
)

In [0]:
# Create complete set of careunit dates

activity_keys = (
    daily_presence_summary
    .select(
        "activity_date",
        "careunit"
    )
    .union(
        daily_arrivals.select(
            "activity_date",
            "careunit"
        )
    )
    .union(
        daily_departures.select(
            "activity_date",
            "careunit"
        )
    )
    .dropDuplicates(
        ["activity_date", "careunit"]
    )
)

# Combine presence and movement measures

daily_operational_activity = (
    activity_keys

    .join(
        daily_presence_summary,
        on=["activity_date", "careunit"],
        how="left"
    )

    .join(
        daily_arrivals,
        on=["activity_date", "careunit"],
        how="left"
    )

    .join(
        daily_departures,
        on=["activity_date", "careunit"],
        how="left"
    )

    .fillna(
        0,
        subset=[
            "active_patient_count",
            "distinct_admission_count",
            "patient_arrival_count",
            "admission_arrival_count",
            "transfer_arrival_count",
            "patient_departure_count"
        ]
    )

    .withColumn(
        "net_patient_flow",
        F.col("patient_arrival_count")
        - F.col("patient_departure_count")
    )

    .withColumn(
        "total_patient_movement",
        F.col("patient_arrival_count")
        + F.col("patient_departure_count")
    )
)

display(
    daily_operational_activity.agg(
        F.count("*").alias("careunit_day_rows"),

        F.countDistinct(
            "activity_date",
            "careunit"
        ).alias("unique_careunit_days"),

        F.countDistinct("careunit")
            .alias("distinct_careunits"),

        F.sum("patient_arrival_count")
            .alias("total_arrivals"),

        F.sum("patient_departure_count")
            .alias("total_departures"),

        F.sum(
            F.when(
                F.col("active_patient_count") == 0,
                1
            ).otherwise(0)
        ).alias("movement_only_careunit_days"),

        F.min("activity_date")
            .alias("first_activity_date"),

        F.max("activity_date")
            .alias("last_activity_date")
    )
)

## Map ICU Stays to Operational Careunit Intervals

The `transfers` table remains the canonical source for careunit location.

ICU stays are mapped to valid transfer intervals using hospital admission ID and overlapping timestamps. Only the overlapping portion of each ICU stay and transfer interval is used.

In [0]:
# Prepare ICU intervals for overlap matching

icu_intervals = (
    icustays
    .filter(
        F.col("stay_id").isNotNull()
        & F.col("hadm_id").isNotNull()
        & F.col("intime").isNotNull()
    )
    .select(
        "subject_id",
        "hadm_id",
        "stay_id",
        F.col("intime").alias("icu_intime"),
        F.col("outtime").alias("icu_outtime")
    )
    .withColumn(
        "icu_effective_outtime",
        F.when(
            F.col("icu_outtime").isNull(),
            F.to_timestamp(
                F.date_add(
                    F.to_date("icu_intime"),
                    1
                )
            )
        ).otherwise(
            F.col("icu_outtime")
        )
    )
)

# Match ICU stays to operational transfer intervals

icu_transfer_overlap = (
    operational_intervals.alias("t")
    .join(
        F.broadcast(icu_intervals).alias("i"),
        (
            (F.col("t.hadm_id") == F.col("i.hadm_id"))
            & (F.col("t.intime") < F.col("i.icu_effective_outtime"))
            & (F.col("t.effective_outtime") > F.col("i.icu_intime"))
        ),
        how="inner"
    )
    .withColumn(
        "overlap_start",
        F.greatest(
            F.col("t.intime"),
            F.col("i.icu_intime")
        )
    )
    .withColumn(
        "overlap_end",
        F.least(
            F.col("t.effective_outtime"),
            F.col("i.icu_effective_outtime")
        )
    )
    .filter(
        F.col("overlap_end") > F.col("overlap_start")
    )
    .select(
        F.col("i.subject_id").alias("subject_id"),
        F.col("i.hadm_id").alias("hadm_id"),
        F.col("i.stay_id").alias("stay_id"),
        F.col("t.transfer_id").alias("transfer_id"),
        F.col("t.careunit").alias("careunit"),
        "overlap_start",
        "overlap_end"
    )
)

# Expand ICU overlap into daily operational activity

icu_operational_daily = (
    icu_transfer_overlap
    .withColumn(
        "start_date",
        F.to_date("overlap_start")
    )
    .withColumn(
        "end_date",
        F.to_date(
            F.expr("overlap_end - INTERVAL 1 SECOND")
        )
    )
    .withColumn(
        "activity_date",
        F.explode(
            F.sequence(
                F.col("start_date"),
                F.col("end_date")
            )
        )
    )
    .select(
        "subject_id",
        "hadm_id",
        "stay_id",
        "careunit",
        "activity_date"
    )
    .dropDuplicates(
        ["stay_id", "careunit", "activity_date"]
    )
)

display(
    icu_operational_daily.agg(
        F.count("*")
            .alias("icu_operational_unit_days"),

        F.countDistinct("stay_id")
            .alias("mapped_icu_stays"),

        F.countDistinct(
            "stay_id",
            "activity_date"
        ).alias("mapped_icu_stay_days"),

        F.countDistinct("careunit")
            .alias("mapped_careunits"),

        F.min("activity_date")
            .alias("first_activity_date"),

        F.max("activity_date")
            .alias("last_activity_date")
    )
)

In [0]:
# Aggregate ICU activity to careunit-day grain

daily_icu_operational_activity = (
    icu_operational_daily
    .groupBy(
        "activity_date",
        "careunit"
    )
    .agg(
        F.countDistinct("stay_id")
            .alias("active_icu_stay_count"),

        F.countDistinct("subject_id")
            .alias("active_icu_patient_count")
    )
)

# Add ICU activity to operational backbone

operational_with_icu = (
    daily_operational_activity
    .join(
        daily_icu_operational_activity,
        on=["activity_date", "careunit"],
        how="left"
    )
    .fillna(
        0,
        subset=[
            "active_icu_stay_count",
            "active_icu_patient_count"
        ]
    )
    .withColumn(
        "icu_activity_flag",
        F.when(
            F.col("active_icu_stay_count") > 0,
            1
        ).otherwise(0)
    )
)

display(
    operational_with_icu.agg(
        F.count("*").alias("careunit_day_rows"),

        F.countDistinct(
            "activity_date",
            "careunit"
        ).alias("unique_careunit_days"),

        F.countDistinct("careunit")
            .alias("distinct_careunits"),

        F.sum("icu_activity_flag")
            .alias("careunit_days_with_icu_activity"),

        F.sum("active_icu_stay_count")
            .alias("mapped_icu_unit_day_activity"),

        F.min("activity_date")
            .alias("first_activity_date"),

        F.max("activity_date")
            .alias("last_activity_date")
    )
)

In [0]:
# Add derived operational measures

gold_activity_base = (
    operational_with_icu
    .withColumn(
        "daily_patient_turnover",
        F.when(
            F.col("active_patient_count") > 0,
            F.col("total_patient_movement")
            / F.col("active_patient_count")
        ).otherwise(
            F.lit(None).cast("double")
        )
    )
)

display(
    gold_activity_base.agg(
        F.count("*").alias("careunit_day_rows"),

        F.sum(
            F.col("daily_patient_turnover")
            .isNull()
            .cast("int")
        ).alias("null_turnover_days"),

        F.min("daily_patient_turnover")
            .alias("min_patient_turnover"),

        F.max("daily_patient_turnover")
            .alias("max_patient_turnover")
    )
)

In [0]:
# Calculate careunit-specific 95th percentile thresholds

careunit_activity_thresholds = (
    gold_activity_base
    .groupBy("careunit")
    .agg(
        F.expr(
            "percentile_approx(active_patient_count, 0.95)"
        ).alias("p95_active_patient_count"),

        F.expr(
            "percentile_approx(total_patient_movement, 0.95)"
        ).alias("p95_total_patient_movement")
    )
)

In [0]:
# Add dataset-relative high activity indicators

gold_activity_flagged = (
    gold_activity_base
    .join(
        careunit_activity_thresholds,
        on="careunit",
        how="left"
    )
    .withColumn(
        "high_patient_load_flag",
        F.when(
            F.col("active_patient_count")
            > F.col("p95_active_patient_count"),
            1
        ).otherwise(0)
    )
    .withColumn(
        "high_movement_flag",
        F.when(
            F.col("total_patient_movement")
            > F.col("p95_total_patient_movement"),
            1
        ).otherwise(0)
    )
    .withColumn(
        "high_activity_flag",
        F.when(
            (F.col("high_patient_load_flag") == 1)
            | (F.col("high_movement_flag") == 1),
            1
        ).otherwise(0)
    )
)

display(
    gold_activity_flagged.agg(
        F.count("*").alias("careunit_day_rows"),

        F.sum("high_patient_load_flag")
            .alias("high_patient_load_days"),

        F.sum("high_movement_flag")
            .alias("high_movement_days"),

        F.sum("high_activity_flag")
            .alias("high_activity_days"),

        (
            F.sum("high_activity_flag")
            / F.count("*")
        ).alias("high_activity_share")
    )
)

In [0]:
# Attach Gold dimension keys

fact_careunit_daily_activity = (
    gold_activity_flagged.alias("g")

    .join(
        dim_date
        .select(
            F.col("calendar_date").alias("activity_date"),
            "date_key"
        ),
        on="activity_date",
        how="left"
    )

    .join(
        dim_careunit
        .select(
            F.col("careunit_name").alias("careunit"),
            "careunit_key"
        ),
        on="careunit",
        how="left"
    )

    .select(
        "date_key",
        "careunit_key",

        # Natural context
        "activity_date",
        "careunit",

        # Daily presence
        "active_patient_count",
        "distinct_admission_count",

        # Movement
        "patient_arrival_count",
        "admission_arrival_count",
        "transfer_arrival_count",
        "patient_departure_count",
        "net_patient_flow",
        "total_patient_movement",
        "daily_patient_turnover",

        # ICU activity
        "active_icu_patient_count",
        "active_icu_stay_count",
        "icu_activity_flag",

        # Careunit-relative benchmarks
        "p95_active_patient_count",
        "p95_total_patient_movement",
        "high_patient_load_flag",
        "high_movement_flag",
        "high_activity_flag"
    )
)

In [0]:
# Validate final fact before publishing

fact_validation = (
    fact_careunit_daily_activity
    .agg(
        F.count("*")
            .alias("rows"),

        F.countDistinct(
            "activity_date",
            "careunit"
        ).alias("unique_careunit_days"),

        F.countDistinct("careunit")
            .alias("careunits"),

        F.sum("patient_arrival_count")
            .alias("arrivals"),

        F.sum("patient_departure_count")
            .alias("departures"),

        F.sum("high_activity_flag")
            .alias("high_activity_days"),

        F.sum("icu_activity_flag")
            .alias("icu_activity_days"),

        F.sum(
            F.col("date_key").isNull().cast("int")
        ).alias("missing_date_keys"),

        F.sum(
            F.col("careunit_key").isNull().cast("int")
        ).alias("missing_careunit_keys")
    )
)

display(fact_validation)

In [0]:
# Confirm grain and required dimension keys

validation_status = (
    fact_validation
    .withColumn(
        "grain_status",
        F.when(
            F.col("rows") == F.col("unique_careunit_days"),
            "PASS"
        ).otherwise("FAIL")
    )
    .withColumn(
        "dimension_key_status",
        F.when(
            (F.col("missing_date_keys") == 0)
            & (F.col("missing_careunit_keys") == 0),
            "PASS"
        ).otherwise("FAIL")
    )
)

display(validation_status)

In [0]:
# Add Gold audit timestamp and publish

fact_careunit_daily_activity_gold = (
    fact_careunit_daily_activity
    .withColumn(
        "_gold_timestamp",
        F.current_timestamp()
    )
)

(
    fact_careunit_daily_activity_gold
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{GOLD_SCHEMA}.fact_careunit_daily_activity"
    )
)

print(
    f"Published: {GOLD_SCHEMA}.fact_careunit_daily_activity"
)

In [0]:
# Reload persisted Gold fact

persisted_activity_fact = spark.table(
    f"{GOLD_SCHEMA}.fact_careunit_daily_activity"
)

print("Persisted operational activity fact loaded.")

In [0]:
# Validate persisted fact

persisted_validation = (
    persisted_activity_fact
    .agg(
        F.count("*")
            .alias("rows"),

        F.countDistinct(
            "activity_date",
            "careunit"
        ).alias("unique_careunit_days"),

        F.countDistinct("careunit")
            .alias("careunits"),

        F.sum("patient_arrival_count")
            .alias("arrivals"),

        F.sum("patient_departure_count")
            .alias("departures"),

        F.sum("high_activity_flag")
            .alias("high_activity_days"),

        F.sum("icu_activity_flag")
            .alias("icu_activity_days"),

        F.sum(
            F.col("date_key").isNull().cast("int")
        ).alias("missing_date_keys"),

        F.sum(
            F.col("careunit_key").isNull().cast("int")
        ).alias("missing_careunit_keys"),

        F.sum(
            F.col("_gold_timestamp").isNull().cast("int")
        ).alias("missing_gold_timestamps")
    )
    .withColumn(
        "grain_status",
        F.when(
            F.col("rows") == F.col("unique_careunit_days"),
            "PASS"
        ).otherwise("FAIL")
    )
    .withColumn(
        "dimension_key_status",
        F.when(
            (F.col("missing_date_keys") == 0)
            & (F.col("missing_careunit_keys") == 0),
            "PASS"
        ).otherwise("FAIL")
    )
    .withColumn(
        "publish_status",
        F.when(
            F.col("missing_gold_timestamps") == 0,
            "PASS"
        ).otherwise("FAIL")
    )
)

display(persisted_validation)